In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.model_selection import train_test_split
from core.Log import *
from core.plots import *
from core.modelUtils import *
from core.CNNmodel import *
from core.CardiacCTdataset import *
from core.preprocessing import *
import tqdm
import logging
import ast

folds_logger()
root_logger()
logger = logging.getLogger('root')

CNTRL_dicom_root = "../Takotsubo-Syndrome/data/Inputs/normal_cases/"
TTS_dicom_root = "../Takotsubo-Syndrome/data/Inputs/takotsubo_cases/"
root_dir = "data/cases/"



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
folds_logger()
root_logger()
logger = logging.getLogger('root')

full_dl = load_dataset_info()
training_dl = [d for d in full_dl if d['pool']== "training"]   #holdout
training_lbls = np.array([d['label'] for d in training_dl])

logger.info(f"Total Cases: {len(training_lbls)} | TTS: {np.sum(training_lbls == 1)}, Control: {np.sum(training_lbls == 0)}")


2025-08-13 03:19:00,227 | Total Cases: 147 | TTS: 77, Control: 70


## Outer Loop
### For performance estimation.
After completing all 4 outer folds, you will have 4 independent performance scores.
The average of these scores gives a reliable estimate of the model's real-world performance.

Outer Split K = 4:
It divides main training pool into 4 non-overlapping folds.

	- 75% training & hyperparameter tuning for that fold
	
	- 25% hold-out test set to evaluate the model trained for that fold

## Inner Loop
### For hyperparameter tuning
By averaging the validation scores across the 3 inner folds, we can reliably determine which hyperparameter set is the best for that specific outer fold's data

Inner Split K = 3:
It takes the 75% of data from the outer loop and splits it further into 3 non-overlapping folds.

	- 67% (of the 75%) for training (50% of the original pool)
	
	- 33% (of the 75%) for validation (25% of the original pool)


In [ ]:
param_grid = [
	{"Set": 1, "LR": 1e-4, "WD": 1e-5, "DR": 0.3},
	{"Set": 2, "LR": 1e-4, "WD": 1e-5, "DR": 0.5},
	{"Set": 3, "LR": 5e-5, "WD": 1e-6, "DR": 0.4}]
for params in param_grid:
	params.update({"epochs": 50, "patience": 5, "batch_size": 8, "threshold_cutoff": 0.5})
OUTER_FOLD_SCORES = []
OUTER_FOLD_EPOCH_HISTORY = []
OUTER_FOLD_BEST_HYPERPARAMS = []


OUTER_K = 4
INNER_K = 3

OUTER_cv = StratifiedKFold(n_splits=OUTER_K, shuffle=True, random_state=42)
INNER_cv = StratifiedKFold(n_splits=INNER_K, shuffle=True, random_state=42)

logger.info(f"Starting {OUTER_K}-fold Nested CV with a search space of {len(param_grid)} hyperparameters.")
logger.info(f"Starting Outer {OUTER_K}-fold Nested Cross-Validation")

for OUT_K, (OUTER_train_idx, OUTER_test_idx) in tqdm(enumerate(OUTER_cv.split(training_dl, training_lbls))):
	logger.info(f"--- Outer Fold {OUT_K + 1}/{OUTER_K} ---")

	OUTER_train = [training_dl[i] for i in OUTER_train_idx]
	OUTER_test = [training_dl[i] for i in OUTER_test_idx]

	OUTER_train_lbls = [d['label'] for d in OUTER_train]

	# Calculate normalization stats ONLY on this fold's training data
	OUT_K_stats = get_dataset_stats(OUTER_train)
	logger.info(f"Outer Fold {OUT_K + 1} Stats: {OUT_K_stats}")

	val_test_transforms, train_transforms = get_transforms(OUT_K_stats)

	hyperparameter_performance = {}  # To store avg validation score for each hyperparam set
	logger.info(f"Starting Inner {INNER_K} Cross-Validation")
	fold_hyperparameter_scores = []

	for hypers in tqdm(param_grid, desc="Hyperparameter Search", leave=False):
		inner_val_losses = []

		for INNER_K, (INNER_train_idx, INNER_val_idx) in enumerate(INNER_cv.split(OUTER_train, OUTER_train_lbls)):
			train_fold = [OUTER_train[i] for i in INNER_train_idx]
			val_fold = [OUTER_train[i] for i in INNER_val_idx]
			batch = hypers["batch_size"]
			train_dataset = CardiacCTDataset(train_fold, train_transforms, OUT_K_stats)
			val_dataset   = CardiacCTDataset(val_fold, val_test_transforms, OUT_K_stats)

			num_workers = 4
			trn_loader  = DataLoader(train_dataset, batch_size=batch, shuffle=True, num_workers=num_workers)
			val_loader  = DataLoader(val_dataset, batch_size=batch, shuffle=False, num_workers=num_workers)

			device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
			model = MultiViewCNN(dropout_rate=hypers["DR"]).to(device)
			best_val_loss = train_val_model(model, trn_loader, val_loader, hypers)
			inner_val_losses.append(best_val_loss)
		avg_loss = np.mean(inner_val_losses)

		# Average the validation performance for this hyperparameter set across all inner folds
		hyperparameter_performance[str(hypers)] = avg_loss
		logger.info(f"Hyperparams: {hypers} -> Avg Inner Val Loss: {avg_loss:.4f}")

	# 3. SELECT the best hyperparameters for this outer fold
	best_hyperparams_str = min(hyperparameter_performance, key=hyperparameter_performance.get)
	best_hyperparams = ast.literal_eval(best_hyperparams_str) # Safely convert string back to dict
	OUTER_FOLD_BEST_HYPERPARAMS.append(best_hyperparams)

	logger.info(f"Best hyperparameters for Outer Fold {OUT_K + 1}: {best_hyperparams}")
	logger.info("Training final model for this outer fold on all of its training data...")

	# use the last inner-fold's validation set again
	final_train_indices, final_val_indices = list(INNER_cv.split(OUTER_train, OUTER_train_lbls))[-1]
	final_train_data = [OUTER_train[i] for i in final_train_indices]
	final_val_data = [OUTER_train[i] for i in final_val_indices]

	train_dataset = CardiacCTDataset(final_train_data, train_transforms, OUT_K_stats)
	val_dataset   = CardiacCTDataset(final_val_data, val_test_transforms, OUT_K_stats)
	test_dataset   = CardiacCTDataset(OUTER_test, val_test_transforms, OUT_K_stats)

	num_workers = 4
	trn_loader  = DataLoader(train_dataset, batch_size=best_hyperparams["batch_size"], shuffle=True, num_workers=num_workers)
	val_loader  = DataLoader(val_dataset, batch_size=best_hyperparams["batch_size"], shuffle=False, num_workers=num_workers)
	final_test_loader  = DataLoader(test_dataset, batch_size=best_hyperparams["batch_size"], shuffle=False, num_workers=num_workers)

	final_model = MultiViewCNN(dropout_rate=best_hyperparams["DR"])
	trained_model, epoch_history, best_val_loss = outer_train_model(final_model, trn_loader, val_loader, best_hyperparams)

	logger.info("Evaluating final model on the outer test set...")
	final_scores = evaluate_model(trained_model, final_test_loader, best_hyperparams)
	final_scores['outer_fold'] = OUT_K + 1
	final_scores['model'] = 'MultiViewCNN'
	logger.info(f"End of Outer Fold {OUT_K + 1} | Test results ---> {final_scores}")

	OUTER_FOLD_SCORES.append(final_scores)
	OUTER_FOLD_EPOCH_HISTORY.append(epoch_history)

logger.info("\n--- Nested Cross-Validation Complete ---")
logger.info(f"Final scores from all {OUTER_K} folds: {OUTER_FOLD_SCORES}")
# Calculate and log the average performance across all outer folds
avg_accuracy = np.mean([s['accuracy'] for s in OUTER_FOLD_SCORES])
std_accuracy = np.std([s['accuracy'] for s in OUTER_FOLD_SCORES])
avg_auc = np.mean([s['auc'] for s in OUTER_FOLD_SCORES])
std_auc = np.std([s['auc'] for s in OUTER_FOLD_SCORES])

logger.info(f"Average Model Accuracy: {avg_accuracy:.4f} ± {std_accuracy:.4f}")
logger.info(f"Average Model AUC: {avg_auc:.4f} ± {std_auc:.4f}")



2025-08-12 23:09:38,824 | Total Cases: 157 | TTS: 82, Control: 75
2025-08-12 23:09:38,825 | Starting 3-fold Nested Cross-Validation...
2025-08-12 23:09:38,828 | Starting Fold 1 | Data Split: 93 train, 32 val, 32 test.
2025-08-12 23:10:08,974 | Epoch 1: Best Val Loss: inf, Patience: 0/3
2025-08-12 23:10:42,211 | Epoch 2: Best Val Loss: 0.6688, Patience: 0/3
2025-08-12 23:11:17,198 | Epoch 3: Best Val Loss: 0.6663, Patience: 0/3
2025-08-12 23:11:55,242 | Epoch 4: Best Val Loss: 0.6648, Patience: 0/3
2025-08-12 23:12:33,663 | Epoch 5: Best Val Loss: 0.6648, Patience: 1/3
2025-08-12 23:12:51,030 | End of Fold 1 | Test results ---> {'loss': 0.6456846445798874, 'accuracy': 0.5625, 'auc': 0.6549019607843137, 'learning_rate': 0.0001, 'fold_id': '1', 'model_name': 'MultiViewCNN'}
2025-08-12 23:12:51,033 | Starting Fold 2 | Data Split: 93 train, 32 val, 32 test.
2025-08-12 23:13:31,640 | Epoch 1: Best Val Loss: inf, Patience: 0/3
2025-08-12 23:14:10,348 | Epoch 2: Best Val Loss: 0.6706, Patience

In [ ]:

best_fold_index = np.argmax([s['auc'] for s in OUTER_FOLD_SCORES])
best_overall_hyperparams = OUTER_FOLD_BEST_HYPERPARAMS[best_fold_index]
logger.info(f"\n--- Hyperparameter search complete ---")
logger.info(f"Best performing hyperparameters identified: {best_overall_hyperparams}")
final_train_data, final_val_data = train_test_split(
    training_dl,
    test_size=0.1, # Use 10% of the main set for validation
    random_state=42,
    stratify=training_lbls
)
FINAL_stats = get_dataset_stats(final_train_data)
final_test_dl = [d for d in full_dl if d['pool']== "holdout"]
val_test_transforms, train_transforms = get_transforms(FINAL_stats)


train_dataset = CardiacCTDataset(final_train_data, train_transforms, FINAL_stats)
val_dataset   = CardiacCTDataset(final_val_data, val_test_transforms, FINAL_stats)
test_dataset   = CardiacCTDataset(final_test_dl, val_test_transforms, FINAL_stats)

num_workers = 4
trn_loader  = DataLoader(train_dataset, batch_size=best_overall_hyperparams["batch_size"], shuffle=True, num_workers=num_workers)
val_loader  = DataLoader(val_dataset, batch_size=best_overall_hyperparams["batch_size"], shuffle=False, num_workers=num_workers)
final_test_loader  = DataLoader(test_dataset, batch_size=best_overall_hyperparams["batch_size"], shuffle=False, num_workers=num_workers)


2025-08-12 23:20:27,379 | --- Nested Cross-Validation Complete ---
2025-08-12 23:20:27,380 | Final Scores across 3 folds: ['0.6549', '0.6863', '0.6627']
2025-08-12 23:20:27,381 | Average Model Performance: 0.6680 ± 0.0133
